## Principles of Machine Learning Final Project
**By: Saheli Ray, Wyatt Golden, HuiDi Hu**

In [4]:
# All imports needed for the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings("ignore")

pd.options.display.max_columns = 100

## Data Exploration / Cleaning
**In this section of the notebook, we will explore the dataset, seeing what it contains and figure out how we will modify it in order to make it the best for the model**

In [5]:
# Load dataset and look at it's shape and first 5 rows
train = pd.read_csv("train.csv", low_memory=False)
print("Dataset shape:", train.shape)
train.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

## Selected Features for Model Training

This dataset contains many variables, but for this first modeling pass we chose fields that are both predictive and reliable. The target variable `INDICATED_DAMAGE` is also included.

### Kept for the model
- `LATITUDE` / `LONGITUDE`: geographic position gives useful location signal without relying on high-cardinality airport codes.
- `AC_CLASS`, `AC_MASS`, `TYPE_ENG`: aircraft size, class, and engine type are good proxies for vulnerability and damage potential.
- `SPECIES`: the type of wildlife is a strong indicator of the likely impact.
- `SIZE`: animal size is directly related to damage severity.
- `SPEED`: aircraft speed at impact is important for energy transfer and damage likelihood.
- `NUM_STRUCK`: number of animals struck could correlate with damage extent.
- `INDICATED_DAMAGE`: target variable indicating whether damage occurred.

### Dropped by design for now
- `PHASE_OF_FLIGHT`: we decided not to use this for prediction, even though it could influence damage, because it may add noise without a strong signal for classification.
- `HEIGHT`: also omitted to keep the model focused on simpler, more direct predictors.
- `SKY`: weather/sky condition is less reliable and not included in this pass.
- `INCIDENT_MONTH`, `TIME_OF_DAY`: temporal factors may not directly affect damage severity post-strike.

### Other variables not used for this first model
- Record identifiers and timestamps: `INDEX_NR`, `INCIDENT_DATE`, `INCIDENT_YEAR`, `LUPDATE`, `TRANSFER`.
- Airport/location identifiers: `AIRPORT_ID`, `AIRPORT`, `STATE`, `FAAREGION`, `LOCATION`, `ENROUTE_STATE`.
- Operator/aircraft metadata: `OPID`, `OPERATOR`, `REG`, `FLT`, `AIRCRAFT`, `AMA`, `AMO`, `EMA`, `EMO`.
- Engine configuration details: `NUM_ENGS`, `ENG_1_POS`, `ENG_2_POS`, `ENG_3_POS`, `ENG_4_POS`.

Most of these fields are excluded because they are either high-cardinality, too sparse, redundant with stronger features, or more appropriate for later experimentation once the baseline model is stable.

- Other strike context: `RUNWAY`, `DISTANCE`, `PRECIPITATION`, `BIRD_BAND_NUMBER`, `SPECIES_ID`, `OUT_OF_RANGE_SPECIES`, `REMARKS`, `REMAINS_COLLECTED`, `REMAINS_SENT`, `WARNED`, `NUM_SEEN`, `SOURCE`, `PERSON`.

In [30]:
# The feature values we will be using to train our model, plus the target.
selected_columns = [
    "LATITUDE",
    "LONGITUDE",
    "AC_CLASS",
    "AC_MASS",
    "TYPE_ENG",
    "SPEED",
    "SPECIES",
    "SIZE",
    "NUM_STRUCK",
    "INDICATED_DAMAGE",  # Target variable
]

train_trimmed = train[selected_columns].copy()

# Take a look at the trimmed dataset's shape and first 5 rows
print(train_trimmed.shape)
print(train_trimmed.head())

# Save this trimmed dataset to a new CSV file for easier access in the future
train_trimmed.to_csv("train_trimmed.csv", index=False)

(307178, 10)
   LATITUDE   LONGITUDE AC_CLASS  AC_MASS TYPE_ENG  SPEED  \
0  18.43942   -66.00183      A        4.0        D  145.0   
1  2.745578  101.709917      A        4.0        D    NaN   
2  38.17439     -85.736      A        4.0        D  240.0   
3  33.94254  -118.40807      NaN      NaN      NaN    NaN   
4  21.97598  -159.33896      A        4.0        D  135.0   

                 SPECIES    SIZE NUM_STRUCK  INDICATED_DAMAGE  
0   Unknown bird - small   Small     10-Feb                 0  
1  Unknown bird - medium  Medium          1                 0  
2   Unknown bird - large   Large          1                 1  
3           Western gull  Medium     10-Feb                 0  
4      American barn owl  Medium          1                 0  


## Target Variable Distribution Comparison

Before dropping rows with missing values, let's compare the target variable distribution between the full dataset and the subset that will remain after cleaning. This ensures we're not introducing bias by dropping missing data.

In [1]:
print("Target distribution before dropping rows with missing values in any feature:")
print(train_trimmed['INDICATED_DAMAGE'].value_counts(normalize=True) * 100)
print()

# Drop rows with missing values in all selected features
features_to_check = [col for col in selected_columns]
train_clean = train_trimmed.dropna(subset=features_to_check)

print("Target distribution after dropping rows with missing values in any feature:")
print(train_clean['INDICATED_DAMAGE'].value_counts(normalize=True) * 100)
print()

print(f"Rows before dropping missing features: {len(train_trimmed)}")
print(f"Rows after dropping missing features: {len(train_clean)}")
print(f"Rows dropped: {len(train_trimmed) - len(train_clean)}")

# Save this trimmed dataset to a new CSV file for easier access in the future
train_clean.to_csv("train_clean.csv", index=False)

Target distribution before dropping rows with missing values in any feature:


NameError: name 'train_trimmed' is not defined

**Since we still have a lot of records to train on and the class distriution actually seemed to improve, we think it is safe to continue with just dropping the missing values**